# Tuki v3: enseñar a conversar al modelo v2 sin perder la traducción — Kaggle

Parte del modelo **v2** (el que ya usa la app para traducir) y lo ajusta con tutorías + repaso de traducción.

**Antes de ejecutar**
1. *Add Input → Upload* → sube `gemma_v3_dataset.zip` como dataset (Kaggle lo descomprime solo).
2. *Add Input → Your Work → Notebooks* → agrega la **versión terminada del cuaderno con que entrenaste el v2**.
   Debe contener la carpeta `gemma_aikukisna/` (el modelo v2 fusionado). Si la borraste de su Output, avísale a Claude.
3. *Add-ons → Secrets*: activa `HF_TOKEN` para este cuaderno.
4. Panel derecho → *Session options*: **Accelerator: GPU T4 x2** e **Internet: On**.

**Cómo ejecutarlo:** *Save Version → Save & Run All (Commit)*. Unas 5–7 h en segundo plano. Al terminar, en **Output**:
`evaluacion_prueba.json` (antes = v2, después = v3), `muestras_tutoria.json` y la carpeta `gemma_aikukisna_v3/`.


In [ ]:
!pip -q install -U "transformers>=4.50" "peft>=0.14" "accelerate>=1.3" datasets sacrebleu sentencepiece
# Colab trae torchao 0.10, que peft rechaza; el entrenamiento no lo necesita. Se desinstala DESPUÉS
# de instalar lo demás, para que ninguna dependencia lo vuelva a traer.
!pip -q uninstall -y torchao
import importlib.util
assert importlib.util.find_spec("torchao") is None, "torchao sigue instalado: reinicia la sesión y repite esta celda"

In [ ]:
import os
# Se usa una sola T4: con las dos, el Trainer activa DataParallel, que choca con gradient checkpointing.
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
from huggingface_hub import login
from kaggle_secrets import UserSecretsClient
try:
    login(UserSecretsClient().get_secret("HF_TOKEN").strip())
except Exception as e:  # no se necesita para entrenar desde el v2
    print("AVISO: no se pudo iniciar sesión en Hugging Face:", type(e).__name__, "- se sigue sin token.")
    print("Antes de la conversión, revisa el secret HF_TOKEN (token tipo Read, sin espacios).")

In [ ]:
import glob, json, os, shutil, zipfile
CARPETA = "/kaggle/working"
os.makedirs("datos", exist_ok=True)
ARCHIVOS = ("entrenamiento.jsonl", "validacion.jsonl", "prueba.jsonl", "muestras_tutoria.jsonl")
zips = glob.glob("/kaggle/input/**/gemma_v3_dataset.zip", recursive=True)
if zips:
    zipfile.ZipFile(zips[0]).extractall("datos")
else:
    marcas = glob.glob("/kaggle/input/**/muestras_tutoria.jsonl", recursive=True)
    if not marcas:
        for raiz, _, nombres in os.walk("/kaggle/input"):
            if any(n.endswith((".jsonl", ".zip")) for n in nombres):
                print(raiz, sorted(n for n in nombres if n.endswith((".jsonl", ".json", ".zip"))))
        raise SystemExit("No encuentro el dataset v3 (muestras_tutoria.jsonl). Arriba se listan los Inputs: sube "
                         "gemma_v3_dataset.zip con Add Input → Upload (o sus 4 .jsonl juntos en un mismo dataset).")
    origen = os.path.dirname(marcas[0])
    print("Dataset v3 en", origen)
    for nombre in ARCHIVOS + ("informe.json",):
        if os.path.exists(os.path.join(origen, nombre)):
            shutil.copy(os.path.join(origen, nombre), "datos")
faltan = [n for n in ARCHIVOS if not os.path.exists(os.path.join("datos", n))]
assert not faltan, f"Faltan en el dataset v3: {faltan}"
print(os.listdir("datos"))
informe = json.load(open("datos/informe.json")) if os.path.exists("datos/informe.json") else {}
print("Informe:", informe)


## 1. Cargar modelo y datos

In [ ]:
import json, random, torch
from transformers import AutoTokenizer, AutoModelForCausalLM

configs = glob.glob("/kaggle/input/**/gemma_aikukisna/config.json", recursive=True)
assert configs, ("No encuentro el modelo v2 (carpeta gemma_aikukisna/): Add Input → Your Work → Notebooks → "
                 "la versión terminada del cuaderno con que entrenaste el v2")
BASE = os.path.dirname(configs[0])  # modelo v2 fusionado: se parte de él, no de google/gemma-3-1b-it
print("Partiendo de", BASE)
tok = AutoTokenizer.from_pretrained(BASE)
# fp16 desborda con Gemma 3 en T4: se entrena en fp32 (1B cabe de sobra en 16 GB con LoRA).
dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float32
modelo = AutoModelForCausalLM.from_pretrained(BASE, torch_dtype=dtype, attn_implementation="eager").to("cuda")

def leer(nombre):
    with open(f"datos/{nombre}", encoding="utf8") as f:
        return [json.loads(l) for l in f if l.strip()]

entrenamiento, validacion, prueba = leer("entrenamiento.jsonl"), leer("validacion.jsonl"), leer("prueba.jsonl")
print(len(entrenamiento), len(validacion), len(prueba))

In [ ]:
# Misma plantilla que GemmaTraductorLocal en la app.
def prompt(texto):
    return f"<start_of_turn>user\n{texto}<end_of_turn>\n<start_of_turn>model\n"

# El prompt de tutoría de la app mide ~400 tokens: con 160 se cortaba antes de la respuesta.
MAX_LEN = 640

def codificar(ej):
    p = tok(prompt(ej["prompt"]), add_special_tokens=False)["input_ids"]
    r = tok(ej["respuesta"] + "<end_of_turn>", add_special_tokens=False)["input_ids"]
    ids = ([tok.bos_token_id] + p + r)[:MAX_LEN]
    # Solo se aprende la respuesta, no la instrucción.
    etiquetas = ([-100] * (1 + len(p)) + r)[:MAX_LEN]
    return {"input_ids": ids, "labels": etiquetas}

from datasets import Dataset
ds_entrenamiento = Dataset.from_list(entrenamiento).map(codificar, remove_columns=list(entrenamiento[0].keys()))
ds_validacion = Dataset.from_list(validacion[:400]).map(codificar, remove_columns=list(validacion[0].keys()))

## 2. Medir el modelo v2 (antes del ajuste)

In [ ]:
import sacrebleu
from collections import defaultdict

@torch.no_grad()
def traducir(textos, lote=16, max_nuevos=64):
    tok.padding_side = "left"
    salidas = []
    for i in range(0, len(textos), lote):
        entrada = tok([tok.bos_token + prompt(t) for t in textos[i:i + lote]], return_tensors="pt",
                      padding=True, add_special_tokens=False).to("cuda")
        gen = modelo.generate(**entrada, max_new_tokens=max_nuevos, do_sample=False)
        for fila in gen[:, entrada["input_ids"].shape[1]:]:
            salidas.append(tok.decode(fila, skip_special_tokens=True).split("<end_of_turn>")[0].strip())
    return salidas

def evaluar(ejemplos, por_direccion=60):
    grupos = defaultdict(list)
    for e in ejemplos:
        grupos[e["direccion"]].append(e)
    resultado = {}
    for direccion, filas in sorted(grupos.items()):
        filas = filas[:por_direccion]
        hip = traducir([f["prompt"] for f in filas])
        resultado[direccion] = round(sacrebleu.corpus_chrf(hip, [[f["respuesta"] for f in filas]], word_order=2).score, 1)
        print(direccion, resultado[direccion], "| ej:", filas[0]["prompt"].split("\n\n")[1], "→", hip[0])
    return resultado

ARCHIVO_ANTES = f"{CARPETA}/evaluacion_base.json"
if os.path.exists(ARCHIVO_ANTES):
    antes = json.load(open(ARCHIVO_ANTES))  # ya medido en una sesión anterior
else:
    antes = evaluar(prueba)
    json.dump(antes, open(ARCHIVO_ANTES, "w"), indent=2)

## 3. Ajuste fino con LoRA

In [ ]:
from peft import LoraConfig, get_peft_model
from transformers import Trainer, TrainingArguments, DataCollatorForSeq2Seq

modelo.gradient_checkpointing_enable()
modelo.enable_input_require_grads()
modelo = get_peft_model(modelo, LoraConfig(
    r=16, lora_alpha=32, lora_dropout=0.05, task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
))
modelo.print_trainable_parameters()

args = TrainingArguments(
    output_dir=f"{CARPETA}/lora", num_train_epochs=1, per_device_train_batch_size=4,
    # warmup_steps en vez de warmup_ratio: transformers 5 eliminó warmup_ratio (30 ≈ 3 % de ~940 pasos).
    gradient_accumulation_steps=8, learning_rate=1e-4, lr_scheduler_type="cosine", warmup_steps=30,
    logging_steps=50, per_device_eval_batch_size=1, eval_strategy="steps", eval_steps=500, save_steps=250, save_total_limit=2,
    bf16=dtype == torch.bfloat16, report_to="none",
)
entrenador = Trainer(
    model=modelo, args=args, train_dataset=ds_entrenamiento, eval_dataset=ds_validacion,
    data_collator=DataCollatorForSeq2Seq(tok, padding=True, label_pad_token_id=-100),
)
import glob, math, shutil
def es_de_esta_corrida(ruta):
    # El Input del v2 también puede traer lora/checkpoint-*: solo vale un punto con los mismos pasos totales.
    estado = os.path.join(ruta, "trainer_state.json")
    if not os.path.exists(estado):
        return False
    pasos = math.ceil(len(ds_entrenamiento) / (args.per_device_train_batch_size * args.gradient_accumulation_steps))
    return json.load(open(estado)).get("max_steps") == pasos
if not glob.glob(f"{CARPETA}/lora/checkpoint-*"):
    previos = [p for p in glob.glob("/kaggle/input/**/lora/checkpoint-*", recursive=True) if es_de_esta_corrida(p)]
    if previos:
        ultimo = max(previos, key=lambda r: int(r.rsplit("-", 1)[1]))
        shutil.copytree(ultimo, f"{CARPETA}/lora/{os.path.basename(ultimo)}")
        print("Punto de control de una corrida anterior:", ultimo)
import glob
puntos = glob.glob(f"{CARPETA}/lora/checkpoint-*")
if puntos:
    print("Retomando desde", max(puntos, key=lambda r: int(r.rsplit("-", 1)[1])))
entrenador.train(resume_from_checkpoint=bool(puntos))

## 4. Medir después del ajuste

In [ ]:
modelo.eval()
despues = evaluar(prueba)
print("\nchrF++ por dirección (antes → después)")
for d in sorted(despues):
    print(f"{d}: {antes.get(d)} → {despues[d]}")
json.dump({"antes": antes, "despues": despues}, open("evaluacion_prueba.json", "w"), indent=2)

In [ ]:
# Cómo conversa: respuestas a tutorías que no vio en el entrenamiento.
muestras = leer("muestras_tutoria.jsonl")
generadas = traducir([m["prompt"] for m in muestras], lote=4, max_nuevos=160)
salida = []
for m, g in zip(muestras, generadas):
    datos = m["prompt"].split("Datos verificados")[-1].split("\n\n")[0]
    salida.append({"pregunta": m["prompt"].rsplit("\n\n", 1)[-1], "datos": "Datos verificados" + datos,
                   "esperada": m["respuesta"], "generada": g})
json.dump(salida, open("muestras_tutoria.json", "w", encoding="utf8"), ensure_ascii=False, indent=1)
for s in salida[:8]:
    print("P:", s["pregunta"], "\n  esperada:", s["esperada"], "\n  generada:", s["generada"], "\n")

## 5. Fusionar LoRA y guardar el modelo completo

In [ ]:
fusionado = modelo.merge_and_unload()
fusionado.save_pretrained("gemma_aikukisna_v3", safe_serialization=True)
tok.save_pretrained("gemma_aikukisna_v3")
del modelo, fusionado, entrenador
torch.cuda.empty_cache()

## 7. Descargar resultados

En **Output** revisa `evaluacion_prueba.json` (antes = v2, después = v3) y `muestras_tutoria.json`, y pásaselos a Claude.
Solo si el v3 traduce casi igual que el v2 (español → Miskitu ≥ 29) se convierte con el cuaderno
`convertir_gemma_aikukisna_v3_kaggle.ipynb` y se publica como Release `modelo-tuki-v3`.

Para ahorrar espacio puedes borrar `lora/` de Output; **no borres** `gemma_aikukisna_v3/` (lo usa la conversión).
